In [3]:
pip install azure-ai-textanalytics azure-core python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# Load environment variables from .env file
load_dotenv()

endpoint = os.getenv("AZURE_LANGUAGE_ENDPOINT")
key = os.getenv("AZURE_LANGUAGE_KEY")

# Initialize the client
client = TextAnalyticsClient(
    endpoint=endpoint, 
    credential=AzureKeyCredential(key)
)

# Test with a simple sentiment analysis call
documents = [
    "The service was excellent.",
    "The delivery was very slow.",
    "The product is acceptable."
]

results = client.analyze_sentiment(documents=documents)

for text, result in zip(documents, results):
    if not result.is_error:
        print(text, "->", result.sentiment)

The service was excellent. -> positive
The delivery was very slow. -> negative
The product is acceptable. -> positive


In [23]:
import os
from dotenv import load_dotenv

from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError

# 1. Read Azure credentials from environment variables
load_dotenv()
endpoint = os.environ["VISION_ENDPOINT"]
key = os.environ["VISION_KEY"]

# 2. Connect to the Azure Computer Vision resource
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# 3. Specify the local image path
image_path = "sample.jpg"

try:
    # 4. Read image bytes
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()

    # 5. Send the image to Azure for Analysis
    result = client.analyze(
        image_data=image_data,
        visual_features=[
            VisualFeatures.CAPTION,
            VisualFeatures.READ
        ],
        gender_neutral_caption=True
    )

    # 6. Output Caption Results
    print("\n--- Image Recognition Results ---")

    if result.caption:
        print("\nImage Description:")
        print(result.caption.text)
        print(f"Confidence: {result.caption.confidence:.2%}")

    # 7. Extract text from the image using OCR
    if result.read:
        print("\nText Detected in Image:")

        for block in result.read.blocks:
            for line in block.lines:
                print(line.text)

    print("\nImage analysis completed successfully.")

except FileNotFoundError:
    print(f"Image file not found: {image_path}")
except HttpResponseError as e:
    print(f"Azure API call failed: {e}")


--- Image Recognition Results ---

Image Description:
a couple of kids playing football
Confidence: 73.09%

Text Detected in Image:
dobit Stock
Adobe Stock | #211758680

Image analysis completed successfully.
